# Publish a Model: Serving in Python

**Working Example.** Copy this file, rename it, and modify *your* copy.

- Author: Denise Case
- Date: 2026-06
- Dataset: Seaborn Penguins
- Target: species
  
Run all cells top to bottom (**Run All**) before pushing to GitHub.

## M6. Serving

This notebook trains a model, saves it, reloads it, and 
tests the **serving core** - the function a web server would call - in-process. 

It does **not** launch a blocking server inside the notebook; 
the real server is shown for you to run with `uv`. 

The analyst decides the API contract and how to handle bad input.

Modules 3-5 built and evaluated models.

This module **publishes** one: 

- a saved artifact and 
- a testable prediction function that a server wraps.

## Overview

This project uses the penguins dataset.
We choose to predict the target `species`.
This target is a **discrete category**, so we have a:

- supervised ML problem (because we've chosen a target)
- a classification problem (because our target is a category)

Customize the overview in your copy to reflect your dataset and choices.

## A. Prepare the Project Environment (.venv/)

- Open one project in VS Code at a time.
- Prepare the .venv/: specify Python version and install / upgrade dependencies listed in `pyproject.toml`.
- Open an integrated terminal (PowerShell if Windows) in the **root project** folder and run:

```shell
uv self update
uv python pin 3.14
uv lock --upgrade
uv sync --extra dev --extra docs --upgrade
```


## B. Select the Notebook Kernel

- Click on the **Select Kernel** name in the top-right corner of the notebook interface.
- Choose Python Environments... /
- Choose the recommended local .venv/ from the drop-down menu.
- This will create a new kernel for the notebook and allow the notebook to use packages installed in the .venv/ environment.

## C. Working in Notebooks (Custom Notes)

- To run a cell, press **Ctrl+Enter** (or **Cmd+Enter** on Mac) when done editing the cell.
- Change the type of a cell (e.g., code or markdown) by looking in the lower left corner of the notebook interface.
- Rearrange cells by dragging and dropping them within the notebook.

See [Run Jupyter Notebooks](https://denisecase.github.io/pro-analytics-02/workflow-b-apply-example-project/run-notebook/) for:

- how to **copy a notebook**
- how to release a `project.log` file
- how to deal with a **stuck kernel**
- etc.

## Section 0. Publishing a Model

A trained model is useful if something can call it on new data. 

Publishing it means three things:

1. **Persist** the trained model to a file (here, with `joblib`) so you do not retrain
   on every request, and reload it the same way.
2. **A serving core** - a small, pure function that takes inputs, validates them,
   calls the model, and returns a result. Keeping this logic in one tested function
   (separate from the web framework) is what makes it reliable and easy to test.
3. **A server** - a thin web layer (here, FastAPI) that receives a request, calls the
   serving core, and returns the result as JSON. The notebook tests the core; the
   server is run separately with `uv`.

The judgment is the **contract**: 

what inputs are required, 
what happens on missing or malformed input, and 
what shape the response takes. 

A server that crashes on bad input is fragile.

## Section 1. Project Setup and Imports

In [ ]:
# === Section 1a. DECLARE IMPORTS ===

from importlib.metadata import version  # to verify
import logging  # for type hinting
import platform  # to verify
from typing import Any  # for type hinting

from datafun_toolkit.logger import get_logger, log_header
import joblib
import pandas as pd

from mlstudio.model_builder_case import (
    DATASET_NAME,
    MODEL_PATH,
    TARGET_COL,
    load_data,
    save_model,
    split_data,
    summarize,
    train_model,
)
from mlstudio.serve_case import predict_from_features

# === Section 1b. CONFIGURE LOGGER ONCE PER NOTEBOOK ===

LOG: logging.Logger = get_logger("M06", level="DEBUG")
log_header(LOG, "M06")


# === Section 1c. USE THE LOGGER TO VERIFY IMPORTS ===

# If any do NOT return a version number, then that package is not installed correctly.
# Check your pyproject.toml and re-run environment setup commands.

LOG.info("Confirming installation:")
LOG.info(f"  python:       {platform.python_version()}")
LOG.info(f"  pandas:       {version('pandas')}")

# === Section 1d. SET PANDAS DISPLAY CONFIGURATION (helps in notebooks) ===

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# === Section 1e. GLOBAL CONSTANTS AND CONFIGURATION ===

# CUSTOM: where the published model artifact is written.
LOG.info(f"Model artifact will be saved to: {MODEL_PATH}")

## Section 2. Load and Prepare the Model

In [ ]:
# === Section 2. Load the Data ===

LOG.info(f"Loading dataset: {DATASET_NAME}")
df_model: pd.DataFrame = load_data()
LOG.info(f"Model rows: {df_model.shape[0]}")
LOG.info(f"Classes in '{TARGET_COL}': {sorted(df_model[TARGET_COL].unique())}")

## Section 3. Split into Train and Test

In [ ]:
# === Section 3. Split into Train and Test ===

X_train, X_test, y_train, y_test = split_data(df_model)
LOG.info(f"Train instances: {len(X_train)}")
LOG.info(f"Test instances:  {len(X_test)}")

## Section 4. Train, Save, Reload Model 

In [ ]:
# === Section 4. Train, Save, and Reload ===

model = train_model(X_train, y_train)
save_model(model)
model = joblib.load(MODEL_PATH)
LOG.info(f"Reloaded model from: {MODEL_PATH}")

## Section 5. Test the Serving Core

In [ ]:
# === Section 5. Test the Serving Core ===

# valid payload - should return a prediction
good_payload: dict[str, Any] = {
    "bill_length_mm": 45.0,
    "bill_depth_mm": 17.0,
    "flipper_length_mm": 200.0,
    "body_mass_g": 4000.0,
}
result: dict[str, Any] = predict_from_features(model, good_payload)
LOG.info(f"Valid payload -> {result}")

# invalid payload - should raise a clean ValueError, not crash
bad_payload: dict[str, Any] = {"bill_length_mm": 45.0}
try:
    predict_from_features(model, bad_payload)
    LOG.warning("Expected a ValueError for the bad payload but none was raised.")
except ValueError as exc:
    LOG.info(f"Invalid payload handled cleanly -> ValueError: {exc}")

## Section 6. Summary and Next Steps

First, output key information (may use Python)
Second, provide your narrative, conclusions, and next steps (in Markdown)

In [ ]:
# === Section 6. Summary ===

# Python summary
summarize()


### Custom Narrative

Summarize your work in this Markdown cell in your notebook.

YOUR JUDGMENT (include a summary in docs/index.md also):

- What is your input contract, and what does bad input return?")
- What does the response include (label only? probabilities?)?")
- How would you confirm the served model matches the saved one?")

### Next Steps

Summarize your next steps in this Markdown cell in your notebook.


## Task: Make the Notebook Yours (Apply / Extend / Explore)

This is an example.
Copy this notebook and make it your own. 

In your copy:

1. At the beginning, update the Author, the purpose, the target, etc. 
2. Remove any instructions you do not need. 

Try things like the following.

1. **Apply** - Add the predicted class **probabilities** to the response
   (`model.predict_proba`) and decide whether a server should expose them.
2. **Extend** - Tighten the contract: reject features outside the ranges seen in
   training, and return a helpful message. Decide where validation belongs - the
   serving core or the web layer.
3. **Explore** - Actually run the FastAPI server with `uv` and POST a few payloads
   (valid and invalid). Record the responses and status codes in `docs/index.md`.

## Task: Final Check

- `README.md` - reflects your description, instructions, commands, and links to your executed notebook.
- `docs/index.md` - reflects your project-specific updates.
- Your GitHub **About** section has a link to your hosted documentation site.
- The executed example notebook AND your custom notebook are available in `notebooks/`.
- Keep this **working example** alongside your custom work until your work has been assessed.
- Ensure your **custom notebook** introduces and narrates **your** custom project.

## Reminder: Run All before pushing to GitHub

Before saving a notebook (and running git add-commit-push), click 'Run All' to generate all outputs and display them in the notebook.

Follow our [pro-analytics-02](https://denisecase.github.io/pro-analytics-02/) common workflows.

Your README.md should have a description, a link to your executed notebook, and a list of commands (updated as you add your custom description, instructions, and commands).

Your docs/ folder should document your custom project analysis in the `docs/index.md` summary.
